# BMBT tier-3: Banglish transliteration model (from scratch)

Trains the tier-3 model for the Banglish pipeline (`bntok.banglish`), on Colab's free T4 tier.
No pretrained checkpoint anywhere - random init, trained here.

Checkpoints go to your mounted Drive, so this survives Colab's ~90min-idle / ~12hr session limits:
just re-run this notebook and training resumes from the last checkpoint automatically.

Before running: set `DRIVE_DATA_DIR` and `DRIVE_CKPT_DIR` below to folders in your own Drive.
Upload `artifacts/banglish-translit-data/{train.tsv,dev.tsv,vocab.json}` (from the repo, already
assembled by `scripts/assemble_banglish_translit_dataset.py`) into `DRIVE_DATA_DIR` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_DATA_DIR = '/content/drive/MyDrive/banglish-translit-data'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/banglish-translit-ckpt'

import os
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'train.tsv')), (
    f'Upload train.tsv/dev.tsv/vocab.json to {DRIVE_DATA_DIR} first '
    '(from artifacts/banglish-translit-data/ in the repo).'
)

In [ ]:
!git clone --depth 1 https://github.com/konkomaji/bornomala.git
%cd bornomala/bengali-tokenizer
!pip install -e . -q

## Train

`--max-steps` and `--batch-size` are conservative defaults for a T4; raise `--batch-size` if you
have headroom (check `nvidia-smi`). Re-running this cell (or the whole notebook) resumes from the
latest checkpoint in `DRIVE_CKPT_DIR` automatically - no flag needed.

In [ ]:
!python scripts/train_banglish_translit.py \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt-dir "{DRIVE_CKPT_DIR}" \
  --device cuda \
  --batch-size 256 \
  --max-steps 20000 \
  --save-every 500 \
  --eval-every 500 \
  --log-every 50

## Evaluate

Against Dakshina's reserved lexicon TEST split - real held-out data, never touched by training or
the tier-1 lookup table. Two numbers: strict word-level exact-match, and character error rate
(partial credit for close-but-wrong). Needs `dakshina_dataset_v1.0/bn` uploaded to Drive too, or
downloaded fresh here (official source, verified reachable - see docs/known-issues.md).

In [ ]:
import os
DAKSHINA_DIR = '/content/dakshina_dataset_v1.0/bn'
if not os.path.exists(DAKSHINA_DIR):
    !curl -s -o /content/dakshina_v1.0.tar https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
    !tar -xf /content/dakshina_v1.0.tar -C /content dakshina_dataset_v1.0/bn/lexicons
    !rm /content/dakshina_v1.0.tar

In [ ]:
import glob
latest_ckpt = sorted(
    glob.glob(os.path.join(DRIVE_CKPT_DIR, 'step-*.pt')),
    key=lambda p: int(p.split('step-')[-1].split('.pt')[0]),
)[-1]
print('evaluating', latest_ckpt)

!python scripts/eval_banglish_translit.py \
  --dakshina-dir "{DAKSHINA_DIR}" \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt "{latest_ckpt}" \
  --device cuda

## Bring the checkpoint home

The checkpoint is already in Drive (`DRIVE_CKPT_DIR`) - download `step-<N>.pt` and `vocab.json`
from there, drop them into `artifacts/banglish-translit-model/` in the repo, and wire the trained
model into `bntok.banglish.transliterate()`'s `tier3_fn` slot (the hook already exists and is
tested - see `tests/test_banglish.py`).